In [3]:
# ==========================================
# MODULE 4: MODELING & PREDICTION
# Linear Regression (Stock Returns Model)
# ==========================================

import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ------------------------------------------
# 1️⃣ LOAD DATA
# ------------------------------------------

df = pd.read_csv("../data/processed/features_dataset.csv", low_memory=False)
df.columns = df.columns.str.strip()
df["Date"] = pd.to_datetime(df["Date"])
print([col for col in df.columns if "return" in col])
list(df)

['return_1d', 'return_5d', 'return_10d', 'return_21d', 'risk_adjusted_return']


/var/folders/kr/8p0vbgj13wvcz3hlcpdwqkh40000gn/T/ipykernel_9400/2731590610.py:19: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df["Date"] = pd.to_datetime(df["Date"])


['Ticker',
 'Date',
 'Close',
 'Open',
 'High',
 'Low',
 'return_1d',
 'return_5d',
 'return_10d',
 'return_21d',
 'ma_5',
 'ma_10',
 'ma_21',
 'price_ma5_ratio',
 'price_ma10_ratio',
 'price_ma21_ratio',
 'vol_5',
 'vol_10',
 'vol_21',
 'daily_range',
 'range_pct',
 'momentum_5',
 'momentum_10',
 'momentum_21',
 'momentum_5_pct',
 'momentum_10_pct',
 'momentum_21_pct',
 '52w_position',
 'gap',
 'risk_adjusted_return',
 'volume_avg_5',
 'volume_vol_5',
 'volume_momentum_5',
 'volume_momentum_5_pct',
 'volume_avg_10',
 'volume_vol_10',
 'volume_momentum_10',
 'volume_momentum_10_pct',
 'volume_avg_21',
 'volume_vol_21',
 'volume_momentum_21',
 'volume_momentum_21_pct',
 'volume_price_ratio',
 'ema_5',
 'ema_10',
 'ema_21',
 'ema_12',
 'ema_26',
 'macd',
 'macd_signal',
 'rsi_14',
 'close_ema5_ratio',
 'close_ema10_ratio',
 'close_ema21_ratio',
 'volume_ma5_ratio',
 'volume_ma10_ratio',
 'volume_ma21_ratio']

In [4]:

# ==========================================
# 2️⃣ LOAD + CLEAN
# ==========================================

df = df.copy()
df = df.dropna(subset=[target_col])
df = df.sort_values(["Ticker", "Date"])

# ==========================================
# 3️⃣ TIME SPLIT
# ==========================================

split_date = df["Date"].quantile(0.8)

train_df = df[df["Date"] <= split_date]
test_df  = df[df["Date"] > split_date]

X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

# ==========================================
# 4️⃣ SCALE + TRAIN RIDGE MODEL
# ==========================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = Ridge(alpha=1.0)
model.fit(X_train_scaled, y_train)

# ==========================================
# 5️⃣ PREDICT
# ==========================================

y_pred = model.predict(X_test_scaled)

# ==========================================
# 6️⃣ MODEL EVALUATION
# ==========================================

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n📊 MODEL PERFORMANCE")
print(f"RMSE: {rmse:.6f}")
print(f"MAE : {mae:.6f}")
print(f"R²  : {r2:.4f}")

# ==========================================
# 7️⃣ FEATURE IMPORTANCE
# ==========================================

coef_df = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient": model.coef_
}).sort_values(by="Coefficient", ascending=False)

print("\n📈 TOP POSITIVE DRIVERS:")
print(coef_df.head(10))

print("\n📉 TOP NEGATIVE DRIVERS:")
print(coef_df.tail(10))

# ==========================================
# 8️⃣ BACKTEST SETUP
# ==========================================

test_df = test_df.copy().reset_index(drop=True)

test_df["Predicted_Return"] = y_pred

# ==========================================
# 9️⃣ THRESHOLD OPTIMIZATION
# ==========================================

thresholds = np.linspace(0, 0.01, 25)

best_sharpe = -np.inf
best_threshold = 0

for t in thresholds:
    signal = (test_df["Predicted_Return"] > t).astype(int)
    strat_ret = signal * test_df[target_col]

    if strat_ret.std() == 0:
        continue

    sharpe = (strat_ret.mean() / strat_ret.std()) * np.sqrt(252)

    if sharpe > best_sharpe:
        best_sharpe = sharpe
        best_threshold = t

print(f"\n🔥 Best Threshold: {best_threshold:.5f}")
print(f"🔥 Best Sharpe (opt): {best_sharpe:.2f}")

# ==========================================
# 🔟 FINAL STRATEGY
# ==========================================

test_df["Signal"] = (test_df["Predicted_Return"] > best_threshold).astype(int)

test_df["Strategy_Return"] = test_df["Signal"] * test_df[target_col]

test_df["Cumulative_Market"] = (1 + test_df[target_col]).cumprod()
test_df["Cumulative_Strategy"] = (1 + test_df["Strategy_Return"]).cumprod()

# ==========================================
# 🔥 ADVANCED RISK METRICS
# ==========================================

test_df["Rolling_Max"] = test_df["Cumulative_Strategy"].cummax()
test_df["Drawdown"] = (
    test_df["Cumulative_Strategy"] - test_df["Rolling_Max"]
) / test_df["Rolling_Max"]

test_df["Volatility_21"] = test_df["Strategy_Return"].rolling(21).std()

test_df["Sharpe_21"] = (
    test_df["Strategy_Return"].rolling(21).mean() /
    test_df["Strategy_Return"].rolling(21).std()
) * np.sqrt(252)

test_df["Win"] = (test_df["Strategy_Return"] > 0).astype(int)
test_df["Trade"] = test_df["Signal"]

# ==========================================
# 📊 FINAL METRICS
# ==========================================

total_return = test_df["Cumulative_Strategy"].iloc[-1] - 1
market_return = test_df["Cumulative_Market"].iloc[-1] - 1

trades = test_df["Trade"].sum()
win_rate = test_df["Win"].sum() / trades if trades > 0 else 0

sharpe = (
    test_df["Strategy_Return"].mean() /
    test_df["Strategy_Return"].std()
) * np.sqrt(252)

max_dd = test_df["Drawdown"].min()

print("\n📊 FINAL STRATEGY PERFORMANCE")
print(f"Strategy Return: {total_return:.2%}")
print(f"Market Return:   {market_return:.2%}")
print(f"Trades:          {int(trades)}")
print(f"Win Rate:        {win_rate:.2%}")
print(f"Sharpe:          {sharpe:.2f}")
print(f"Max Drawdown:    {max_dd:.2%}")

# ==========================================
# 💾 SAVE FINAL OUTPUT
# ==========================================

final_df = test_df[[
    "Ticker",
    "Date",
    "return_1d",
    "Predicted_Return",
    "Signal",
    "Strategy_Return",
    "Cumulative_Market",
    "Cumulative_Strategy",
    "Drawdown",
    "Sharpe_21",
    "Volatility_21"
]].copy()

final_df.to_csv("../data/processed/linear_model_predictions.csv", index=False)

print("\n✅ DONE: FULL OPTIMIZED QUANT PIPELINE COMPLETE")

NameError: name 'target_col' is not defined

In [18]:
list(final_df)

['Ticker',
 'Date',
 'return_1d',
 'Predicted_Return',
 'Signal',
 'Strategy_Return',
 'Cumulative_Market',
 'Cumulative_Strategy',
 'Drawdown',
 'Sharpe_21',
 'Volatility_21']